In [ ]:
import torch
import numpy as np
from PIL import Image
from transformers import AutoModel, AutoVideoProcessor

MODEL_ID = "facebook/vjepa2-vitl-fpc64-256"
NUM_FRAMES = 16
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoVideoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
).to(device)
model.eval()

def get_vjepa_embeddings(pil_image):
    inputs = processor(videos=[pil_image], return_tensors="pt")
    pv = inputs["pixel_values_videos"]
    pv = pv.repeat(1, NUM_FRAMES, 1, 1, 1)
    pv = pv.to(device=device, dtype=torch.float16)
    
    with torch.no_grad():
        outputs = model(
            pixel_values_videos=pv,
            skip_predictor=True,
            output_hidden_states=True
        )
    
    return outputs.last_hidden_state

Using device: cuda


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [20]:
from projectaria_tools.core import data_provider
from projectaria_tools.core.stream_id import StreamId
import pandas as pd
import os

vrs_path = '/mnt/data/home/zj2433/Projects/Ego/CV4Egocentric/data/clean_0_data_unzipped/AriaGen2PilotDataset_v1.0_clean_0_main_recording.vrs'

provider = data_provider.create_vrs_data_provider(vrs_path)

eye_gaze_stream = StreamId("373-1")

num_samples = provider.get_num_data(eye_gaze_stream)
print(f"Total samples: {num_samples}")

# Inspectează primul sample ca să vedem câmpurile
first = provider.get_eye_gaze_data_by_index(eye_gaze_stream, 0)
print("Tip data:", type(first))
print("Atribute:", [m for m in dir(first) if not m.startswith('__')])

Total samples: 9907
Tip data: <class '_core_pybinds.mps.EyeGaze'>
Atribute: ['_pybind11_conduit_v1_', 'combined_gaze_origin_in_cpf', 'combined_gaze_valid', 'depth', 'pitch', 'pitch_high', 'pitch_low', 'session_uid', 'spatial_gaze_point_in_cpf', 'spatial_gaze_point_valid', 'tracking_timestamp', 'vergence', 'yaw', 'yaw_high', 'yaw_low']


[ProgressLogger][INFO]: 2026-04-27 19:19:19: Opening /mnt/data/home/zj2433/Projects/Ego/CV4Egocentric/data/clean_0_data_unzipped/AriaGen2PilotDataset_v1.0_clean_0_main_recording.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/mnt/data/home/zj2433/Projects/Ego/CV4Egocentric/data/clean_0_data_unzipped/AriaGen2PilotDataset_v1.0_clean_0_main_recording.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 211-1/camera-et-left activated
[VrsDataProvider][INFO]: streamId 211-2/camera-et-right activated
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][INFO]: streamId 231-1/mic activated
[VrsDataProvider][INFO]: Fail to activate streamId 240-1
[VrsDataProvider][INFO]: streamId 246-1/temperature activated
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][INFO]: streamId 248-1/ppg activated
[VrsDataProvider][INFO]: streamId 281-1/gps-app activated
[VrsDataProvider][INFO]: streamId 281-2/gps activated
[VrsDataProvider][INFO]: st

In [21]:
records = []
for i in range(num_samples):
    obs = provider.get_eye_gaze_data_by_index(eye_gaze_stream, i)
    records.append({
        'timestamp_ns':                obs.tracking_timestamp.total_seconds() * 1e9,
        'yaw':                         obs.yaw,
        'pitch':                       obs.pitch,
        'yaw_low':                     obs.yaw_low,
        'yaw_high':                    obs.yaw_high,
        'pitch_low':                   obs.pitch_low,
        'pitch_high':                  obs.pitch_high,
        'depth':                       obs.depth,
        'vergence':                    obs.vergence,
        'combined_gaze_valid':         obs.combined_gaze_valid,
        'spatial_gaze_point_valid':    obs.spatial_gaze_point_valid,
        'session_uid':                 obs.session_uid,
    })

import pandas as pd, os
df = pd.DataFrame(records)
print(df.head())
print(f"Shape: {df.shape}")

out_file = '/mnt/data/home/zj2433/Projects/Ego/CV4Egocentric/data/clean_0_data_unzipped/eye_gaze/eye_gaze.csv'
os.makedirs(os.path.dirname(out_file), exist_ok=True)
df.to_csv(out_file, index=False)
print(f"wrote in: {out_file}")

   timestamp_ns       yaw     pitch  yaw_low  yaw_high  pitch_low  pitch_high  \
0  1.199934e+12  0.059639  0.102024      0.0       0.0        0.0         0.0   
1  1.199967e+12  0.066944 -0.029387      0.0       0.0        0.0         0.0   
2  1.200000e+12  0.114963  0.107263      0.0       0.0        0.0         0.0   
3  1.200034e+12  0.003000 -0.189634      0.0       0.0        0.0         0.0   
4  1.200067e+12 -0.006440 -0.185158      0.0       0.0        0.0         0.0   

   depth                                           vergence  \
0    0.0  <_core_pybinds.mps.EyeGazeVergence object at 0...   
1    0.0  <_core_pybinds.mps.EyeGazeVergence object at 0...   
2    0.0  <_core_pybinds.mps.EyeGazeVergence object at 0...   
3    0.0  <_core_pybinds.mps.EyeGazeVergence object at 0...   
4    0.0  <_core_pybinds.mps.EyeGazeVergence object at 0...   

   combined_gaze_valid  spatial_gaze_point_valid session_uid  
0                 True                     False              
1       